In [7]:
import pandas as pd

df_tempo_processed_csv = pd.read_csv('/content/tempo_preprocessed.csv')
display(df_tempo_processed_csv)

,id_berita,judul_berita,isi_berita,kategori_berita,text,clean_text,tokens
0,2076486,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,TENTARA Nasional Indonesia atau TNI menggelar ...,politik,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...,ketika para jenderal ikut defile di hut ke tni...,"['jenderal', 'defile', 'hut', 'tni', 'tentara'..."
1,2076480,Prabowo Minta Semua Pesantren Didata setelah P...,PRESIDENPrabowoSubianto memerintahkan semua po...,politik,Prabowo Minta Semua Pesantren Didata setelah P...,prabowo minta semua pesantren didata setelah p...,"['prabowo', 'pesantren', 'didata', 'ponpes', '..."
2,2076479,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,PRESIDEN Prabowo Subianto memerintahkan Pangli...,politik,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,prabowo utamakan kompetensi prajurit dibanding...,"['prabowo', 'utamakan', 'kompetensi', 'prajuri..."
3,2076473,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,KEMENTERIAN Komunikasi dan Digital (Kemenkomdi...,politik,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,kemenkomdigi permintaan data ke tiktok hanya u...,"['kemenkomdigi', 'permintaan', 'data', 'tiktok..."
4,2076468,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,MANTAN Presiden Megawati Soekarnoputri dan Jok...,politik,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,megawati dan jokowi tak hadir di hut ke tni di...,"['megawati', 'jokowi', 'hadir', 'hut', 'tni', ..."
...,...,...,...,...,...,...,...
895,2073886,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ADA dua kondisi yang kini melekat padaHarry Ka...,sepakbola,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ketika harry kane memecahkan rekor gol cristia...,"['harry', 'kane', 'memecahkan', 'rekor', 'gol'..."
896,2073880,Peluang Timnas Indonesia Lewati Hadangan Arab ...,PENGAMAT sepak bola Tanah Air Kesit Budi Hando...,sepakbola,Peluang Timnas Indonesia Lewati Hadangan Arab ...,peluang timnas indonesia lewati hadangan arab ...,"['peluang', 'timnas', 'indonesia', 'lewati', '..."
897,2073852,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,"DALAM usia 40 tahun,Cristiano Ronaldomasih mam...",sepakbola,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,seperti apa ketajaman cristiano ronaldo bersam...,"['ketajaman', 'cristiano', 'ronaldo', 'nassr',..."
898,2073819,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,"BADAN sepak bola dunia,FIFA, menjatuhkan sanks...",sepakbola,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,fifa jatuhkan sanksi untuk malaysia dan pemain...,"['fifa', 'jatuhkan', 'sanksi', 'malaysia', 'pe..."


In [8]:
df_tempo_processed_csv['tokens_list'] = df_tempo_processed_csv['tokens'].apply(eval)
df_tempo_processed_csv['tokens_str'] = df_tempo_processed_csv['tokens_list'].apply(lambda x: " ".join(x))

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df_tempo_processed_csv['tokens_str'])

In [14]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(n_components=15, random_state=42)
X_topics = lda_model.fit_transform(X_bow)


In [15]:
terms = vectorizer.get_feature_names_out()
num_top_words = 15

for idx, topic in enumerate(lda_model.components_):
    print(f"\nTopik {idx+1}:")
    print(", ".join([terms[i] for i in topic.argsort()[:-num_top_words - 1:-1]]))


Topik 1:
tiktok, data, digital, komdigi, akun, oktober, sistem, elektronik, pembekuan, pemerintah, aktivitas, september, kementerian, kebakaran, pilihan

Topik 2:
pertandingan, liga, babak, kemenangan, laga, pemain, gol, open, korea, tim, champions, final, indonesia, kalah, menit

Topik 3:
indonesia, games, sea, orang, keluarga, indra, jakarta, oktober, bangunan, pilihan, arya, pemain, pesantren, september, muda

Topik 4:
wisata, penerbangan, mandalika, tiket, dunia, pariwisata, wisatawan, motogp, penutupan, ntb, lombok, partai, kawasan, demokrat, jam

Topik 5:
gaza, israel, trump, kpk, korupsi, hamas, palestina, orang, rencana, haji, tersangka, menteri, negara, uang, nadiem

Topik 6:
gunung, kali, pesisir, gempa, september, lewotobi, erupsi, aktivitas, lakilaki, meter, kim, bagas, oktober, kota, pusat

Topik 7:
indonesia, persen, triliun, saham, keuangan, oktober, september, bank, laut, pasar, samudra, barat, gula, miliar, indeks

Topik 8:
jakarta, indonesia, air, oktober, lingkungan

In [19]:
df_topics = pd.DataFrame(X_topics, columns=[f"topik_{i+1}" for i in range(lda_model.n_components)])

# 2️⃣ Gabungkan dengan data awal
df_final = pd.concat([df_tempo_processed_csv, df_topics], axis=1)

# 3️⃣ Tentukan topik dominan
df_final["topik_dominan"] = df_topics.idxmax(axis=1).apply(lambda x: int(x.split("_")[1]))

# 4️⃣ Tampilkan contoh hasil
print("\n=== Contoh 10 Data dengan Topik Dominan ===")
print(df_final[["judul_berita", "kategori_berita", "topik_dominan"]].head(10))

# 5️⃣ Distribusi dokumen per topik
print("\n=== Distribusi Dokumen per Topik ===")
print(df_final["topik_dominan"].value_counts())


=== Contoh 10 Data dengan Topik Dominan ===
                                        judul_berita kategori_berita  \
0  Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI         politik   
1  Prabowo Minta Semua Pesantren Didata setelah P...         politik   
2  Prabowo: Utamakan Kompetensi Prajurit Dibandin...         politik   
3  Kemenkomdigi: Permintaan Data ke TikTok Hanya ...         politik   
4  Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...         politik   
5  Hari Terakhir Pendaftaran: 3,4 Juta Siswa Siap...         politik   
6  Prabowo dan Gibran Kompak Pakai Kemeja Safari ...         politik   
7  Puan: TNI Harus Kuat Hadapi Ancaman, Termasuk ...         politik   
8  Prabowo: Prajurit Berprestasi Berhak Jadi Pemi...         politik   
9  Wapres Gibran Kenakan Safari Krem Dampingi Pra...         politik   

   topik_dominan  
0              9  
1             12  
2              9  
3              1  
4              9  
5             15  
6              9  
7         

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

X = X_topics
y = df_tempo_processed_csv["kategori_berita"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("=== Hasil Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))

=== Hasil Naive Bayes ===
               precision    recall  f1-score   support

      ekonomi       0.38      0.36      0.37        22
      hiburan       0.23      0.23      0.23        22
        hukum       0.19      0.38      0.25        13
internasional       0.60      0.52      0.56        23
   lingkungan       0.67      0.45      0.54        22
     olahraga       0.42      0.61      0.50        18
     otomotif       0.78      0.88      0.82        16
      politik       0.44      0.28      0.34        25
    sepakbola       0.73      0.58      0.65        19

     accuracy                           0.46       180
    macro avg       0.49      0.48      0.47       180
 weighted avg       0.50      0.46      0.47       180



In [21]:
from sklearn import svm
from sklearn.metrics import classification_report

clf = svm.SVC(kernel="linear", random_state=42)
clf.fit(X_train, y_train)
y_pred_svm = clf.predict(X_test)

print("=== Hasil SVM ===")
print(classification_report(y_test, y_pred_svm))

=== Hasil SVM ===
               precision    recall  f1-score   support

      ekonomi       0.39      0.41      0.40        22
      hiburan       0.25      0.27      0.26        22
        hukum       0.23      0.38      0.29        13
internasional       0.60      0.52      0.56        23
   lingkungan       0.65      0.50      0.56        22
     olahraga       0.48      0.78      0.60        18
     otomotif       0.78      0.88      0.82        16
      politik       0.47      0.28      0.35        25
    sepakbola       0.92      0.58      0.71        19

     accuracy                           0.49       180
    macro avg       0.53      0.51      0.51       180
 weighted avg       0.53      0.49      0.50       180



In [22]:
from sklearn.model_selection import train_test_split

# Gunakan dataframe yang benar
X = X_topics
y = df_tempo_processed_csv["kategori_berita"]

# Split data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


               precision    recall  f1-score   support

      ekonomi       0.38      0.36      0.37        22
      hiburan       0.24      0.23      0.23        22
        hukum       0.20      0.38      0.26        13
internasional       0.64      0.61      0.62        23
   lingkungan       0.67      0.45      0.54        22
     olahraga       0.48      0.72      0.58        18
     otomotif       0.78      0.88      0.82        16
      politik       0.41      0.28      0.33        25
    sepakbola       0.86      0.63      0.73        19

     accuracy                           0.49       180
    macro avg       0.52      0.51      0.50       180
 weighted avg       0.52      0.49      0.49       180



In [24]:
dominant_topics = []

for doc_topics in X_topics:
    # Get the index of the dominant topic for the current document
    dominant_topic_index = doc_topics.argmax()
    # Get the probability of the dominant topic
    dominant_topic_probability = doc_topics[dominant_topic_index]
    dominant_topics.append((dominant_topic_index + 1, round(dominant_topic_probability, 3)))

df_dominant = pd.DataFrame(dominant_topics, columns=["Topik Dominan", "Proporsi"])

# Concatenate with the original dataframe and the topic distribution dataframe
df_result = pd.concat([df_tempo_processed_csv.reset_index(drop=True), df_dominant.reset_index(drop=True), df_topics.reset_index(drop=True)], axis=1)

display(df_result[["judul_berita", "kategori_berita", "Topik Dominan", "Proporsi", "clean_text"]])

,judul_berita,kategori_berita,Topik Dominan,Proporsi,clean_text
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,politik,9,0.949,ketika para jenderal ikut defile di hut ke tni...
1,Prabowo Minta Semua Pesantren Didata setelah P...,politik,12,0.371,prabowo minta semua pesantren didata setelah p...
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,politik,9,0.796,prabowo utamakan kompetensi prajurit dibanding...
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,politik,1,0.698,kemenkomdigi permintaan data ke tiktok hanya u...
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,politik,9,0.995,megawati dan jokowi tak hadir di hut ke tni di...
...,...,...,...,...,...
895,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,sepakbola,2,0.553,ketika harry kane memecahkan rekor gol cristia...
896,Peluang Timnas Indonesia Lewati Hadangan Arab ...,sepakbola,10,0.839,peluang timnas indonesia lewati hadangan arab ...
897,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,sepakbola,10,0.994,seperti apa ketajaman cristiano ronaldo bersam...
898,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,sepakbola,3,0.818,fifa jatuhkan sanksi untuk malaysia dan pemain...
